In [1]:
"""
Extract numbered clinical research questions from a PDF into a JSON dataset.

Usage:
    python extract_questions.py input.pdf output.json

The PDF is expected to contain sections like:

    Cardiology
    1. In adults with resistant hypertension, does renal denervation compared with
    optimized medical therapy reduce 24-hour ambulatory systolic blood
    pressure?
    2. In adults with HFrEF ...

Questions can wrap across multiple lines, and section headers (e.g. "Cardiology",
"Neurology") appear on their own line right before a new specialty's questions
start. This script:
  1. Extracts raw text from the PDF (page by page).
  2. Detects section headers (a line with no leading number, no question mark,
     and title-cased / short — matched against a known specialty list, with a
     fallback heuristic).
  3. Detects the start of a new question via a leading "N." pattern.
  4. Joins wrapped lines until the question mark that ends the question.
  5. Because this document contains TWO separate lists of 100 questions each
     (the numbering restarts at 1), a "set" number is tracked so IDs stay
     unique across both lists.
  6. Outputs a JSON list of records: {id, set, serial_number, section, question}
"""

'\nExtract numbered clinical research questions from a PDF into a JSON dataset.\n\nUsage:\n    python extract_questions.py input.pdf output.json\n\nThe PDF is expected to contain sections like:\n\n    Cardiology\n    1. In adults with resistant hypertension, does renal denervation compared with\n    optimized medical therapy reduce 24-hour ambulatory systolic blood\n    pressure?\n    2. In adults with HFrEF ...\n\nQuestions can wrap across multiple lines, and section headers (e.g. "Cardiology",\n"Neurology") appear on their own line right before a new specialty\'s questions\nstart. This script:\n  1. Extracts raw text from the PDF (page by page).\n  2. Detects section headers (a line with no leading number, no question mark,\n     and title-cased / short — matched against a known specialty list, with a\n     fallback heuristic).\n  3. Detects the start of a new question via a leading "N." pattern.\n  4. Joins wrapped lines until the question mark that ends the question.\n  5. Because 

In [2]:
!pip install pdfplumber -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 91.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 88.6 MB/s eta 0:00:00


In [3]:
import re
import sys
import json
from pathlib import Path
import pdfplumber

In [4]:
# Known specialty headers that appear in this document (extend as needed).
KNOWN_SECTIONS = {
    "Cardiology", "Neurology", "Oncology", "Gastroenterology", "Nephrology",
    "Rheumatology", "Pulmonology", "Pediatrics", "Emergency Medicine",
    "Orthopedics", "Obs/Gyn", "Obstetrics & Gynecology", "Dermatology",
}

# Matches the start of a new numbered question, e.g. "12." or "12)"
QUESTION_START_RE = re.compile(r"^(\d{1,3})[\.\)]\s+(.*)$")

In [5]:
def extract_lines(pdf_path):
    lines = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text() or ""
            for line in text.split("\n"):
                line = line.strip()
                if line:
                    lines.append(line)
    return lines


def is_section_header(line):
    # Exact match against known specialties, or a short Title-Case line
    # with no digits/question mark (fallback heuristic for unseen specialties).
    stripped = line.strip()
    if stripped in KNOWN_SECTIONS:
        return True
    if (
        len(stripped.split()) <= 4
        and not stripped.endswith("?")
        and not re.search(r"\d", stripped)
        and stripped[0:1].isupper()
    ):
        return True
    return False


def parse_questions(lines):
    records = []
    current_section = None
    current_set = 1
    last_serial = 0

    buffer_serial = None
    buffer_text = []

    def flush():
        nonlocal buffer_serial, buffer_text
        if buffer_serial is not None and buffer_text:
            question = " ".join(buffer_text).strip()
            question = re.sub(r"\s+", " ", question)
            records.append({
                "set": current_set,
                "serial_number": buffer_serial,
                "section": current_section,
                "question": question,
            })
        buffer_serial = None
        buffer_text = []

    for line in lines:
        m = QUESTION_START_RE.match(line)
        if m:
            # New question begins -> flush the previous one first.
            flush()
            serial = int(m.group(1))
            rest = m.group(2)

            # Detect restart of numbering (1 after having seen higher numbers)
            # which signals the start of the second question list/set.
            if serial == 1 and last_serial > 1:
                current_set += 1

            last_serial = serial
            buffer_serial = serial
            buffer_text = [rest]
        elif is_section_header(line):
            # Section header line - flush any pending question, update section.
            flush()
            current_section = line.strip()
        else:
            # Continuation of the current question (wrapped line).
            if buffer_serial is not None:
                buffer_text.append(line)
            # else: stray line before any question starts (e.g. title) -> ignore

    flush()
    return records


def build_dataset(records):
    dataset = []
    for i, rec in enumerate(records, start=1):
        dataset.append({
            "id": i,  # globally unique sequential id across both sets
            "set": rec["set"],
            "serial_number": rec["serial_number"],
            "section": rec["section"],
            "question": rec["question"],
        })
    return dataset

In [6]:
pdf_path = "/content/New_Research_Questions.pdf"
out_path = "/content/new_research_questions.json"

lines = extract_lines(pdf_path)
records = parse_questions(lines)
dataset = build_dataset(records)

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=2, ensure_ascii=False)

print(f"Extracted {len(dataset)} questions -> {out_path}")

Extracted 200 questions -> /content/new_research_questions.json


In [7]:
# quick sanity check
sets = sorted(set(r["set"] for r in dataset))
for s in sets:
    count = sum(1 for r in dataset if r["set"] == s)
    print(f"  Set {s}: {count} questions")

  Set 1: 100 questions
  Set 2: 100 questions
